# Оптимизация рекламной стратегии на Facebook

В данном эксперименте была исследована эффективность новой рекламной стратегии на Facebook. Для работы использованы [данные с Kaggle](https://www.kaggle.com/datasets/furth3r/facebook-ab-test-of-bidding-feature "Перейти на сайт").

*Исходные данные:*
1) Показы(Impression): Количество показов на одно объявление.
2) Клик(Click): Количество кликов по каждому объявлению.
3) Покупка(Purchase): количество товаров, приобретенных после клика.
4) Начисления(Earning): Начисления после покупки.

В качестве результата приведено заключение об эффективности новой стратегии.

## Импорт необходимых библиотек

In [1]:
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import shapiro, mannwhitneyu
import numpy as np

## Загрузка  и проверка данных

In [2]:
control = pd.read_csv('control_group.csv')
test = pd.read_csv('test_group.csv')

In [3]:
control.head(5)

,Impression,Click,Purchase,Earning
0,82529,6090,665,2311
1,98050,3383,315,1743
2,82696,4168,458,1798
3,109914,4911,487,1696
4,108458,5988,441,1544


In [4]:
test.head(5)

,Impression,Click,Purchase,Earning
0,120104,3217,702,1940
1,134776,3635,834,2929
2,107807,3057,423,2526
3,116445,4650,429,2281
4,145083,5201,750,2782


In [5]:
control.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Impression  40 non-null     int64
 1   Click       40 non-null     int64
 2   Purchase    40 non-null     int64
 3   Earning     40 non-null     int64
dtypes: int64(4)
memory usage: 1.4 KB


In [6]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Impression  40 non-null     int64
 1   Click       40 non-null     int64
 2   Purchase    40 non-null     int64
 3   Earning     40 non-null     int64
dtypes: int64(4)
memory usage: 1.4 KB


**Вывод:** Данные успешно загружены в числовом формате, пропусков нет.

## Расчет метрик

Введем и рассчитаем следующие продуктовые метрики:
1) CTR - Кликабельность. Доля кликов от всех показов.
2) CV - Доля покупок от всех кликов.
3) Conversion - Конверсия. Доля покупок от всех показов.
4) Earning rate - Деньги, которые принесло 1 объявление.

В качестве целевой метрики возьмем Earning rate, так как она напрямую показывает финансовый результат. Остальные будут вспомогальными для объяснения итогов эксперимента.

In [7]:
control['CTR'] = control['Click'] / control['Impression']
control['CV'] = control['Purchase'] / control['Click']
control['Conversion'] = control['Purchase'] / control['Impression']
control['Earning_rate'] = control['Earning'] / control['Impression']

In [8]:
test['CTR'] = test['Click'] / test['Impression']
test['CV'] = test['Purchase'] / test['Click']
test['Conversion'] = test['Purchase'] / test['Impression']
test['Earning_rate'] = test['Earning'] / test['Impression']

## Проведение эксперимента

В ходе эксперимента оценим влияние новой стратегии на Earning rate, посчитаем эффект и прирост вспомотельных метрик для интерпретации результатов.

### Анализ распределений групп

Проведем тест Шапиро-Уилка для проверки нормальности распределений каждой группы. Уровень значимости $\alpha$ = 0.05.

In [9]:
st, p_value = shapiro(control['Earning_rate'])
print(f'Контрольная группа. p-value = {p_value:0.4f}')

Контрольная группа. p-value = 0.2704


In [10]:
st, p_value = shapiro(test['Earning_rate'])
print(f'Тестовая группа. p-value = {p_value:0.4f}')

Тестовая группа. p-value = 0.0160


**Вывод:** Тестовая группа имеет распределение отличное от нормального. В связи с этим применим в дальнейшем тест Манна-Уитни для сравнения групп(уровень значимости $\alpha$ так же зафикструем равным 0.05).

### A\A-тест

Разделим контрольную группу на 2 равные части и сравним их через тест Манна-Уитни, чтобы убедиться в корректности проведения эксперимента.

In [11]:
control_1, control_2 = np.array_split(control['Earning_rate'], 2)

c:\Users\vorpv\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\_core\fromnumeric.py:57: FutureWarning: 'Series.swapaxes' is deprecated and will be removed in a future version. Please use 'Series.transpose' instead.
  return bound(*args, **kwds)


In [12]:
u, p_value = mannwhitneyu(control_1, control_2, alternative='two-sided')
print(f'p-value = {p_value:0.4f}')

p-value = 0.9676


**Вывод:** Тест ожидаемо показал отсутствие статистических различий. Это значит, что используемые методы вызывают доверие.

### A\B-тест

Таким же способом сравним всю контрольную и тестовую группы. В качестве альтернативной гипотезы возьмем увеличение целевой метрики в тестовой группе.

In [13]:
u, p_value = mannwhitneyu(test['Earning_rate'], control['Earning_rate'], alternative='greater')
print(f'p-value = {p_value:0.4f}')

p-value = 0.0235


In [14]:
print(f'Разница Earning rate(размер эффекта): {test['Earning_rate'].median() - control['Earning_rate'].median()}')
print(f'Процентная разница Earning rate: {test['Earning_rate'].median() / control['Earning_rate'].median() - 1:0.2%}')

Разница Earning rate(размер эффекта): 0.0013527085649746966
Процентная разница Earning rate: 7.22%


**Вывод:** Тест указал в пользу альтернативной гипотезы: Earning rate имеет тенденцию к увеличению в тестовой группе по сравнению с контрольной. Медианный Earning rate при этом увеличился на 7.22% - это неплохой результат.

### Оценка вспомогательных метрик

Теперь посчитаем процентный прирост остальных метрик.

In [15]:
for metric in ['CTR', 'CV', 'Conversion']:
    metric_diff = test[metric].median() / control[metric].median() - 1
    print(f'{metric}: {metric_diff:0.2%}')

CTR: -35.74%
CV: 33.47%
Conversion: -11.22%


**Вывод:** Кликабельность упала почти на 36% и конверсия на 11.22%, но при этом доля покупок среди кликов выросла почти на 33.5%. Это говорит о том, что новая стратегия стала более таргетированной, так как обьявления подходят тем, кому действительно интересен товар.

### Анализ воронок продаж

Посмотрим наглядно путь клиентов от просмотра обьявления до покупки.

In [16]:
data = dict(
    number=[control['Impression'].sum(),
            control['Click'].sum(),
            control['Purchase'].sum()
            ],
    stage=["Просмотр", "Клик", "Покупка"]
)

fig = go.Figure(go.Funnel(
    y=data["stage"],
    x=data["number"],
    textinfo="value+percent initial"
))

fig.update_layout(title="Воронка продаж контрольной группы")
fig.show()

In [17]:
data = dict(
    number=[test['Impression'].sum(),
            test['Click'].sum(),
            test['Purchase'].sum()
            ],
    stage=["Просмотр", "Клик", "Покупка"]
)

fig = go.Figure(go.Funnel(
    y=data["stage"],
    x=data["number"],
    textinfo="value+percent initial"
))

fig.update_layout(title="Воронка продаж тестовой группы")
fig.show()

**Вывод:** Как было указано ранее, до непосредсвенной покупки доходит меньший процент людей, но при этом кликнувшие по объявлению чаще покупают товар(14.7% против 10.8%). Воронки подтверждают предположение о большей таргетированности рекламы.

## Выводы

Было выявлено, что при новой рекламной стратегии 1 объявление приносит на 7.22% больше денег. Также стоит отметить, что не смотря на существенное уменьшение конверсии и кликабельности(11.22% и 36% соответственно), реклама стала более таргетированной(на 33.5% больше кликнувших действительно купили товар).

**Итого:**

Новую рекламную стратегию следует внедрять на платформе.